# Fongbe Challenge — Éval CPU des modèles ASR Fongbe existants

Zero-shot (sans fine-tuning) sur ~100 extraits `recommended_validation`.
Runtime : **CPU** (pas de GPU nécessaire, ~18 modèles × 100 clips).

Prérequis : le dossier `fongbe-challenge` partagé sur Drive (raccourci dans My Drive).

## 0. Settings

In [ ]:
from pathlib import Path

DATA_DIR = Path('/content/fongbe_eval')
DATA_DIR.mkdir(parents=True, exist_ok=True)
REPO_RAW = 'https://raw.githubusercontent.com/Appolinairee/fongbe-challenge/master'
GCS_BASE = 'https://storage.googleapis.com/fongbe'
TAR_PREFIXES = ['audio_train_reliable_']
N_EXAMPLES = 100
print('settings ok')

## 1. CSV + scripts depuis Drive

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil
try:
    drive.mount('/content/drive')
except ValueError:
    pass

DRIVE_DATA = Path('/content/drive/MyDrive/fongbe-challenge/data')
for f in DRIVE_DATA.iterdir():
    dst = DATA_DIR / f.name
    if f.suffix in ('.csv', '.py', '.md') and not dst.exists():
        shutil.copy2(f, dst)
        print('✓', f.name)
print('csv ok:', len(list(DATA_DIR.glob('*.csv'))))

## 2. Télécharger + extraire Train_reliable (~13 GB WAV)

In [ ]:
import subprocess, shutil, json, urllib.request

TAR_DIR = DATA_DIR / 'tars'
TAR_DIR.mkdir(exist_ok=True)
listing = json.loads(urllib.request.urlopen(
    'https://storage.googleapis.com/storage/v1/b/fongbe/o?maxResults=200&fields=items(name,size)').read())
all_tars = sorted(i['name'] for i in listing['items'] if i['name'].endswith('.tar'))
tars = [t for t in all_tars if any(t.startswith(p) for p in TAR_PREFIXES)]
print(f'{len(tars)} archives')

for name in tars:
    tar_path = TAR_DIR / name
    if not tar_path.exists():
        print(f'download {name}', flush=True)
        subprocess.run(['wget','-q','-O',str(tar_path),f'{GCS_BASE}/{name}'], check=True)
    print(f'extract {name}', flush=True)
    subprocess.run(['tar','-xf',str(tar_path),'-C',str(DATA_DIR)], check=True)
    tar_path.unlink()
    print(f'ok {name}', flush=True)
print('disque libre:', round(shutil.disk_usage('/content').free/1e9,1), 'GB')

In [ ]:
# Vérif rapide
import pandas as pd
from pathlib import Path
df = pd.read_csv(DATA_DIR/'Train_reliable.csv')
missing = (~df.audio_path.map(lambda p: (DATA_DIR/p).exists())).sum()
print(f'{len(df):,} lignes, {missing} wav manquants')

## 3. Dépendances + script d'éval

In [ ]:
import subprocess, sys, urllib.request
subprocess.run([sys.executable,'-m','pip','install','-q',
    'transformers','soundfile','rapidfuzz==3.14.5','librosa','speechbrain'], check=True)
urllib.request.urlretrieve(f'{REPO_RAW}/scripts/eval_fongbe_models.py',
                           '/content/eval_fongbe_models.py')
print('ok')

## 4. Lancer l'éval (tous les modèles, N=100)

In [ ]:
# Tous les modèles du registre — peut prendre 1-3 h sur CPU
# Pour tester un seul modèle : --models fonxlsr
!python /content/eval_fongbe_models.py \
    --data-dir /content/fongbe_eval \
    --n 100 \
    --out /content/fongbe_eval/eval_models_results.csv

## 5. Résultats

In [ ]:
import pandas as pd
res = pd.read_csv('/content/fongbe_eval/eval_models_results.csv')
print(res.sort_values('score').to_string(index=False))